# درس ۱۶ - استقرار عوامل مقیاس‌پذیر با Microsoft Foundry

در این دفترچه یادداشت شما یک **عامل پشتیبانی مشتری آماده برای تولید** برای شرکت فرضی **Contoso** می‌سازید. برخلاف درس‌های قبلی، نکته اینجا حلقه استدلال عامل نیست — همه چیز *دور* آن است که یک عامل را برای اجرا در مقیاس ایمن می‌کند:

۱. **فراخوانی ابزار** — جستجوی سفارشات و ایجاد تیکت.
۲. **RAG** — پاسخ‌های سیاست از یک پایگاه دانش.
۳. **حافظه** — به خاطر سپردن مشتری در طول تعامل‌ها.
۴. **مسیر‌یابی مدل** — ارسال درخواست‌های ساده به مدل کوچک، و درخواست‌های پیچیده به مدل بزرگ.
۵. **کش کردن پاسخ** — پاسخ‌گویی به سوالات تکراری بدون فراخوانی مدل.
۶. **تایید انسانی** — بازپرداخت‌ها بالاتر از حد آستانه برای تایید متوقف می‌شوند.
۷. **درگاه ارزیابی** — مجموعه تست آفلاینی که انتشار بد را مسدود می‌کند.
۸. **قابلیت مشاهده** — ردیابی OpenTelemetry در اطراف هر درخواست.

هر بخش خودکفا و قابل اجرا است. هر خط را بخوانید — عنصرهای پایه تولید عمداً کوچک نگه داشته شده‌اند.


## راه‌اندازی

قبل از اجرای این دفترچه، اطمینان حاصل کنید که:

1. **یک پروژه Microsoft Foundry** با یک مدل چت مستقر شده دارید (مثلاً `gpt-4.1-mini`).
2. **با Azure CLI وارد شده‌اید** — دستور `az login` را در ترمینال خود اجرا کنید.
3. **متغیرهای محیطی مورد نیاز را تنظیم کرده‌اید:**
   - `AZURE_AI_PROJECT_ENDPOINT` — نقطه پایانی پروژه Microsoft Foundry شما.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — نام مدل مستقر شده شما.

بخش RAG از **Azure AI Search** استفاده می‌کند وقتی `AZURE_SEARCH_SERVICE_ENDPOINT` و `AZURE_SEARCH_API_KEY` تنظیم شده باشند، و در غیر این صورت به جستجوی در حافظه داخلی بازمی‌گردد تا دفترچه بدون نیاز به منبع جستجو اجرا شود.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## ۱. ابزارها

ابزارهای تولید کار واقعی را در برابر سیستم‌های واقعی انجام می‌دهند. در اینجا ما یک پایگاه داده سفارش و یک سیستم صدور بلیت را با توابع ساده پایتون شبیه‌سازی می‌کنیم. دکوراتور `@tool` آن‌ها را به عامل معرفی می‌کند.

دقت کنید که `issue_refund` از `approval_mode="always_require"` برای بازپرداخت‌های بالاتر از یک آستانه استفاده می‌کند — این همان ابتدایی انسان در حلقه است که بعداً پیاده‌سازی می‌کنیم.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — پایگاه دانش سیاست‌ها

سوالات مربوط به سیاست‌ها («پنجره بازگشت شما چند روز است؟») باید از منبع معتبری پاسخ داده شوند، نه از حافظه مدل. ما یک پایگاه دانش کوچک را به عنوان ابزار جستجو بسته‌بندی می‌کنیم.

در محیط تولید این ابزار **Azure AI Search** است؛ در اینجا ما یک جستجوی کلمه کلیدی در حافظه ارائه می‌دهیم تا دفترچه یادداشت در هر جایی اجرا شود و هنگام وجود متغیرهای محیطی به صورت خودکار به Azure AI Search تغییر پیدا کند.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## ۳. حافظه

یک عامل پشتیبانی که فراموش می‌کند با چه کسی صحبت می‌کند، یک عامل پشتیبانی بد است. ما یک مخزن پروفایل کوچک برای هر مشتری نگه می‌داریم و یک خلاصه کوتاه را در دستورالعمل‌های عامل وارد می‌کنیم. در محیط تولید این یک سرویس حافظه است (رجوع کنید به درس ۱۳)؛ در اینجا یک دیکشنری الگو را قابل مشاهده می‌کند.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## ۴ و ۵. مسیریابی مدل و کش کردن پاسخ‌ها

دو اهرم هزینه که به یک هندلر درخواست متصل شده‌اند:

- **مسیریابی**: یک طبقه‌بند هیوریستیک ارزان قیمت تصمیم می‌گیرد که آیا درخواست به مدل کوچک یا بزرگ نیاز دارد.
- **کش کردن**: سوالات تکراری نرمال شده مستقیماً از کش بدون فراخوانی مدل پاسخ داده می‌شوند.

طبقه‌بند اینجا عمداً ساده است. در تولید شما آن را در برابر ترافیک اعتبارسنجی می‌کنید و می‌توانید آن را با مسیریاب مدل Foundry جایگزین کنید.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## ۶ و ۸. نماینده، تأیید انسان، و قابلیت مشاهده  

اکنون نماینده را از ابزارهای فوق می‌سازیم و هر درخواست را در یک بخش OpenTelemetry می‌بندیم. تابع `handle_support_request` مدیریت‌کننده درخواست در محیط تولید است: کش → مسیردهی → ردیابی → اجرا → کش.  

تأیید انسان توسط چارچوب مدیریت می‌شود: چون `issue_refund` دارای `approval_mode="always_require"` است، اجرای برنامه متوقف می‌شود و درخواست تأیید را نمایش می‌دهد که ما به‌صورت صریح آن را حل می‌کنیم.  


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## ۷. دروازه ارزیابی

این دروازه انتشار از درس است: یک مجموعه آزمون آفلاین عامل را ارزیابی می‌کند و استقرار تنها در صورت عبور نرخ موفقیت از یک آستانه ادامه می‌یابد. ارزیاب اینجا یک بررسی ساده همپوشانی کلمات کلیدی است تا دفترچه به صورت خودکفا باقی بماند؛ در محیط تولید شما از یک مدل زبانی بزرگ به عنوان داور یا یک ارزیاب چارچوبی استفاده می‌کنید (درس ۱۰ را ببینید).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## کنار هم گذاشتن: یک انتشار شبیه‌سازی شده

سلول زیر کل حلقه‌ای را که درس توضیح می‌دهد نشان می‌دهد: اجرای دروازه ارزیابی، و تنها در صورتی "انتشار" دهید که آن را رد کند. این الگویی است که شما در CI اجرا می‌کنید قبل از اینکه یک نسخه عامل را به سرویس عامل Foundry ارتقا دهید.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## خلاصه

شما یک نماینده پشتیبانی مشتری آماده تولید جمع‌آوری کرده‌اید که هر نگرانی عملیاتی به آن متصل شده است:

- **ابزارها، RAG، و حافظه** به نماینده قابلیت و زمینه می‌دهند.
- **مسیر‌یابی مدل و کشینگ** تأخیر و هزینه را کنترل می‌کند.
- **تأیید انسانی** از اقدامات پرخطر مانند بازپرداخت‌های بزرگ محافظت می‌کند.
- **دروازه ارزیابی** از انتشار نسخه‌های بد قبل از ارسال جلوگیری می‌کند.
- **ردیابی** هر درخواست را قابل مشاهده می‌کند.

### چالش

این نماینده را گسترش دهید تا:

۱. **پشتیبانی از چند مدل** — یک سطح سوم "استدلال" اضافه کنید و تخلفات/شکایات را به آن هدایت کنید.
۲. **افزودن دروازه‌های ارزیابی** — `TEST_CASES` را برای شامل کردن سناریوهای تأیید بازپرداخت گسترش دهید و تأیید کنید که دروازه پسرفت‌ها را می‌گیرد.
۳. **افزودن مسیر‌یابی با آگاهی هزینه** — هزینه تخمینی برای هر درخواست (کوچک در مقابل بزرگ در مقابل کش) را پیگیری کنید و بعد از یک دسته از پرس و جوهای مخلوط گزارش هزینه چاپ کنید.

در درس بعدی، شما مسیر مخالف را طی می‌کنید و نماینده را کاملاً روی دستگاه خود با Microsoft Foundry Local و Qwen اجرا می‌کنید.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
